# Day 09. Exercise 00
# Regularization

## 0. Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier

## 1. Preprocessing

1. Read the file `dayofweek.csv` that you used in the previous day to a dataframe.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [2]:
df = pd.read_csv('../data/dayofweek.csv')
df.head()

,numTrials,hour,dayofweek,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,...,labname_lab02,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1
0,-0.788667,-2.562352,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,-0.756764,-2.562352,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,-0.724861,-2.562352,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,-0.692958,-2.562352,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,-0.661055,-2.562352,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [3]:
X = df.drop(columns=['dayofweek'])
y = df['dayofweek']

random_state = 21

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((1348, 43), (338, 43), (1348,), (338,))

## 2. Logreg regularization

### a. Default regularization

1. Train a baseline model with the only parameters `random_state=21`, `fit_intercept=False`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model


The result of the code where you trained and evaluated the baseline model should be exactly like this (use `%%time` to get the info about how long it took to run the cell):

```
train -  0.62902   |   valid -  0.59259
train -  0.64633   |   valid -  0.62963
train -  0.63479   |   valid -  0.56296
train -  0.65622   |   valid -  0.61481
train -  0.63397   |   valid -  0.57778
train -  0.64056   |   valid -  0.59259
train -  0.64138   |   valid -  0.65926
train -  0.65952   |   valid -  0.56296
train -  0.64333   |   valid -  0.59701
train -  0.63674   |   valid -  0.62687
Average accuracy on crossval is 0.60165
Std is 0.02943
```

In [4]:
def crossval(model, X, y, n_splits):
    scores = []
    skf = StratifiedKFold(n_splits=n_splits)

    for train, valid in skf.split(X, y): 
        X_train, X_valid = X.iloc[train], X.iloc[valid]
        y_train, y_valid = y.iloc[train], y.iloc[valid]

        model.fit(X_train, y_train)
        X_train_score = model.score(X_train, y_train)
        X_valid_score = model.score(X_valid, y_valid)

        scores.append(X_valid_score)

        print(f'train -  {X_train_score:.5f}  |  valid -  {X_valid_score:.5f}')
    print(f"Средняя точность перекрёстной проверки:{np.mean(scores):.5f}\n\
Стандартное отклонение:{np.std(scores):.5f}")

In [5]:
%%time
lr = LogisticRegression(random_state=21, fit_intercept=False)

crossval(lr, X_train, y_train, 10)

train -  0.62819  |  valid -  0.59259
train -  0.64716  |  valid -  0.62963
train -  0.63479  |  valid -  0.57037
train -  0.65540  |  valid -  0.61481
train -  0.63314  |  valid -  0.57778
train -  0.64056  |  valid -  0.59259
train -  0.64221  |  valid -  0.65926
train -  0.65952  |  valid -  0.56296
train -  0.64333  |  valid -  0.59701
train -  0.63591  |  valid -  0.62687
Средняя точность перекрёстной проверки:0.60239
Стандартное отклонение:0.02852
CPU times: user 8.03 s, sys: 204 ms, total: 8.23 s
Wall time: 1.27 s


### b. Optimizing regularization parameters

1. In the cells below try different values of penalty: `none`, `l1`, `l2` – you can change the values of solver too.

In [6]:
%%time
lr = LogisticRegression(random_state=21, fit_intercept=False)

crossval(lr, X_train, y_train, 10)

train -  0.62819  |  valid -  0.59259
train -  0.64716  |  valid -  0.62963
train -  0.63479  |  valid -  0.57037
train -  0.65540  |  valid -  0.61481
train -  0.63314  |  valid -  0.57778
train -  0.64056  |  valid -  0.59259
train -  0.64221  |  valid -  0.65926
train -  0.65952  |  valid -  0.56296
train -  0.64333  |  valid -  0.59701
train -  0.63591  |  valid -  0.62687
Средняя точность перекрёстной проверки:0.60239
Стандартное отклонение:0.02852
CPU times: user 7.8 s, sys: 48.9 ms, total: 7.85 s
Wall time: 1.18 s


In [7]:
%%time
lr = LogisticRegression(solver='saga', penalty='l1', max_iter=1000, random_state=21, fit_intercept=False)

crossval(lr, X_train, y_train, 10)

train -  0.63726  |  valid -  0.58519
train -  0.64221  |  valid -  0.61481
train -  0.62984  |  valid -  0.55556
train -  0.64386  |  valid -  0.60000
train -  0.63232  |  valid -  0.57778
train -  0.63644  |  valid -  0.57778
train -  0.63644  |  valid -  0.65926
train -  0.65622  |  valid -  0.57778
train -  0.64580  |  valid -  0.58955
train -  0.63839  |  valid -  0.62687
Средняя точность перекрёстной проверки:0.59646
Стандартное отклонение:0.02848
CPU times: user 6.11 s, sys: 1.21 ms, total: 6.11 s
Wall time: 5.79 s


In [8]:
%%time
lr = LogisticRegression(solver='lbfgs', penalty='l2', random_state=21, fit_intercept=False)

crossval(lr, X_train, y_train, 10)

train -  0.62819  |  valid -  0.59259
train -  0.64716  |  valid -  0.62963
train -  0.63479  |  valid -  0.57037
train -  0.65540  |  valid -  0.61481
train -  0.63314  |  valid -  0.57778
train -  0.64056  |  valid -  0.59259
train -  0.64221  |  valid -  0.65926
train -  0.65952  |  valid -  0.56296
train -  0.64333  |  valid -  0.59701
train -  0.63591  |  valid -  0.62687
Средняя точность перекрёстной проверки:0.60239
Стандартное отклонение:0.02852
CPU times: user 7.14 s, sys: 3.72 ms, total: 7.14 s
Wall time: 1.03 s


## 3. SVM regularization

### a. Default regularization

1. Train a baseline model with the only parameters `probability=True`, `kernel='linear'`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [9]:
%%time
svc = SVC(kernel='linear', probability=True, random_state=21)

crossval(svc, X_train, y_train, 10)

train -  0.70486  |  valid -  0.65926
train -  0.69662  |  valid -  0.75556
train -  0.69415  |  valid -  0.62222
train -  0.70239  |  valid -  0.65185
train -  0.69085  |  valid -  0.65185
train -  0.68920  |  valid -  0.64444
train -  0.69250  |  valid -  0.72593
train -  0.70074  |  valid -  0.62222
train -  0.69605  |  valid -  0.61940
train -  0.71087  |  valid -  0.63433
Средняя точность перекрёстной проверки:0.65871
Стандартное отклонение:0.04359
CPU times: user 4 s, sys: 3.58 ms, total: 4 s
Wall time: 3.66 s


### b. Optimizing regularization parameters

1. In the cells below try different values of the parameter `C`.

In [10]:
%%time
svc = SVC(kernel='linear', probability=True, random_state=21, C=0.5)

crossval(svc, X_train, y_train, 10)

train -  0.66694  |  valid -  0.63704
train -  0.66612  |  valid -  0.73333
train -  0.67271  |  valid -  0.60741
train -  0.67354  |  valid -  0.62963
train -  0.67766  |  valid -  0.64444
train -  0.66529  |  valid -  0.61481
train -  0.66200  |  valid -  0.68889
train -  0.66529  |  valid -  0.57037
train -  0.67463  |  valid -  0.59701
train -  0.66804  |  valid -  0.61194
Средняя точность перекрёстной проверки:0.63349
Стандартное отклонение:0.04471
CPU times: user 3.6 s, sys: 3.67 ms, total: 3.6 s
Wall time: 3.61 s


In [11]:
%%time
svc = SVC(kernel='linear', probability=True, random_state=21, C=1.5)

crossval(svc, X_train, y_train, 10)

train -  0.70734  |  valid -  0.65926
train -  0.70486  |  valid -  0.74815
train -  0.74361  |  valid -  0.64444
train -  0.70816  |  valid -  0.67407
train -  0.70239  |  valid -  0.67407
train -  0.69909  |  valid -  0.64444
train -  0.70239  |  valid -  0.72593
train -  0.70239  |  valid -  0.62963
train -  0.70840  |  valid -  0.64925
train -  0.71911  |  valid -  0.64179
Средняя точность перекрёстной проверки:0.66910
Стандартное отклонение:0.03679
CPU times: user 3.69 s, sys: 2.84 ms, total: 3.7 s
Wall time: 3.7 s


In [12]:
%%time
svc = SVC(kernel='linear', probability=True, random_state=21, C=3)

crossval(svc, X_train, y_train, 10)

train -  0.71228  |  valid -  0.63704
train -  0.73042  |  valid -  0.77778
train -  0.76834  |  valid -  0.65926
train -  0.73124  |  valid -  0.66667
train -  0.71641  |  valid -  0.70370
train -  0.72712  |  valid -  0.68889
train -  0.71888  |  valid -  0.71852
train -  0.74361  |  valid -  0.62963
train -  0.74629  |  valid -  0.66418
train -  0.72570  |  valid -  0.64925
Средняя точность перекрёстной проверки:0.67949
Стандартное отклонение:0.04227
CPU times: user 3.92 s, sys: 4.75 ms, total: 3.92 s
Wall time: 3.93 s


## 4. Tree

### a. Default regularization

1. Train a baseline model with the only parameter `max_depth=10` and `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [13]:
def crossval(model, X, y, n_splits):
    scores = []
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=21)

    for train, valid in skf.split(X, y): 
        X_train, X_valid = X.iloc[train], X.iloc[valid]
        y_train, y_valid = y.iloc[train], y.iloc[valid]

        model.fit(X_train, y_train)

        X_train_score = model.score(X_train, y_train)
        X_valid_score = model.score(X_valid, y_valid)

        scores.append(X_valid_score)

        print(f'train -  {X_train_score:.5f}  |  valid -  {X_valid_score:.5f}')
    print(f"Средняя точность перекрёстной проверки:{np.mean(scores):.5f}\n\
Стандартное отклонение:{np.std(scores):.5f}")

In [14]:
%%time
dtc = DecisionTreeClassifier(max_depth=10, random_state=21)

crossval(dtc, X_train, y_train, 10)

train -  0.80874  |  valid -  0.77037
train -  0.79802  |  valid -  0.70370
train -  0.81286  |  valid -  0.72593
train -  0.80049  |  valid -  0.74815
train -  0.80956  |  valid -  0.68889
train -  0.78978  |  valid -  0.74074
train -  0.80627  |  valid -  0.60741
train -  0.82688  |  valid -  0.71111
train -  0.78995  |  valid -  0.79104
train -  0.80313  |  valid -  0.70896
Средняя точность перекрёстной проверки:0.71963
Стандартное отклонение:0.04791
CPU times: user 92.9 ms, sys: 0 ns, total: 92.9 ms
Wall time: 92.6 ms


### b. Optimizing regularization parameters

1. In the cells below try different values of the parameter `max_depth`.
2. As a bonus, play with other regularization parameters trying to find the best combination.

In [15]:
param_grid = {
    "max_depth": [10, 15, 20],
    "max_features": ['sqrt', 'log2'],
    "min_samples_split": [2, 3, 4],
    "min_samples_leaf": [1, 2, 3]
    }

dtc = DecisionTreeClassifier(random_state=21)

grid = GridSearchCV(
    dtc,
    param_grid=param_grid,
    cv=10,
    scoring='accuracy'
    )

model_grid = grid.fit(X_train, y_train)

print(f'Лучшие гиперпараметры: {model_grid.best_params_}\nИх точность: {model_grid.score(X_test, y_test)}')

Лучшие гиперпараметры: {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 3}
Их точность: 0.8254437869822485


In [16]:
%%time
dtc = DecisionTreeClassifier(max_depth=20, max_features='sqrt', min_samples_split=2, min_samples_leaf=2, random_state=21)

crossval(dtc, X_train, y_train, 10)

train -  0.79390  |  valid -  0.67407
train -  0.82275  |  valid -  0.72593
train -  0.86480  |  valid -  0.82222
train -  0.85161  |  valid -  0.83704
train -  0.80297  |  valid -  0.69630
train -  0.85161  |  valid -  0.74074
train -  0.84501  |  valid -  0.80000
train -  0.83430  |  valid -  0.75556
train -  0.81713  |  valid -  0.81343
train -  0.78254  |  valid -  0.67910
Средняя точность перекрёстной проверки:0.75444
Стандартное отклонение:0.05787
CPU times: user 76.6 ms, sys: 961 μs, total: 77.5 ms
Wall time: 77.1 ms


## 5. Random forest

### a. Default regularization

1. Train a baseline model with the only parameters `n_estimators=50`, `max_depth=14`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [17]:
rfc = RandomForestClassifier(n_estimators=50, max_depth=14, random_state=21)

crossval(rfc, X_train, y_train, 10)

train -  0.97939  |  valid -  0.85185
train -  0.96620  |  valid -  0.85926
train -  0.96208  |  valid -  0.91852
train -  0.97115  |  valid -  0.91852
train -  0.97197  |  valid -  0.88148
train -  0.96538  |  valid -  0.86667
train -  0.96455  |  valid -  0.88889
train -  0.96867  |  valid -  0.87407
train -  0.96458  |  valid -  0.93284
train -  0.96787  |  valid -  0.86567
Средняя точность перекрёстной проверки:0.88578
Стандартное отклонение:0.02673


### b. Optimizing regularization parameters

1. In the new cells try different values of the parameters `max_depth` and `n_estimators`.
2. As a bonus, play with other regularization parameters trying to find the best combination.

In [18]:
param_grid = {
    "n_estimators": [20, 50, 100],
    "max_depth": [None, 15, 20],
    "max_features": ['sqrt', 'log2'],
    "min_samples_split": [2, 3, 4],
    "min_samples_leaf": [1, 2, 3]
    }

dtc = RandomForestClassifier(random_state=21)

grid = GridSearchCV(
    dtc,
    param_grid=param_grid,
    cv=10,
    scoring='accuracy'
    )

model_grid = grid.fit(X_train, y_train)

print(f'Лучшие гиперпараметры: {model_grid.best_params_}\nИх точность: {model_grid.score(X_test, y_test)}')

Лучшие гиперпараметры: {'max_depth': None, 'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}
Их точность: 0.9378698224852071


In [19]:
rfc = RandomForestClassifier(n_estimators=100, max_depth=None, max_features='log2', min_samples_leaf=1, min_samples_split=2, random_state=21)

crossval(rfc, X_train, y_train, 10)

train -  1.00000  |  valid -  0.91111
train -  1.00000  |  valid -  0.86667
train -  1.00000  |  valid -  0.94815
train -  1.00000  |  valid -  0.91111
train -  1.00000  |  valid -  0.92593
train -  1.00000  |  valid -  0.92593
train -  1.00000  |  valid -  0.93333
train -  1.00000  |  valid -  0.88148
train -  1.00000  |  valid -  0.94776
train -  1.00000  |  valid -  0.85821
Средняя точность перекрёстной проверки:0.91097
Стандартное отклонение:0.03049


## 6. Predictions

1. Choose the best model and use it to make predictions for the test dataset.
2. Calculate the final accuracy.
3. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your test dataset).
4. Save the model.

In [20]:
rfc = RandomForestClassifier(n_estimators=100, max_depth=None, max_features='log2', min_samples_leaf=1, min_samples_split=2, random_state=21)

rfc = rfc.fit(X_train, y_train)
y_pred = rfc.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Итоговая точность наилучшей модели: {accuracy}')

Итоговая точность наилучшей модели: 0.9378698224852071


In [21]:
errors = y_test[y_test != y_pred]
total_per_day = y_test.value_counts()
errors_per_day = errors.value_counts()
error_rate_per_day = (errors_per_day / total_per_day).fillna(0) * 100

print("Ошибка по дням в %:")
print(error_rate_per_day.sort_values(ascending=False))

Ошибка по дням в %:
dayofweek
0    22.222222
4    14.285714
5     7.407407
2     6.666667
3     3.750000
1     3.636364
6     1.408451
Name: count, dtype: float64


In [22]:
import joblib

joblib.dump(rfc, '../data/model_01.pkl')

['../data/model_01.pkl']